
# Bulk RNA-Seq Pipeline Example

This notebook demonstrates a complete workflow for analyzing bulk RNA sequencing (RNA-seq) data, starting from raw FASTQ files and proceeding through alignment, quantification, and downstream analysis using publicly available data.

## Pipeline Overview

The pipeline includes the following steps:

1. **Download raw FASTQ files** from the Sequence Read Archive (SRA) using SRR accession numbers  
2. **Perform read preprocessing and quality control**, including optional trimming and filtering  
3. **Align reads** to the human reference genome  
4. **Quantify gene expression** to generate raw gene-level count matrices  
5. **Perform differential expression analysis** using DESeq2, including normalization and statistical testing  
6. **Assess data quality and sample relationships** using summary statistics and visualization (e.g., PCA, MA, and volcano plots)  
7. **Conduct pathway enrichment analysis** using Gene Set Enrichment Analysis (GSEA)

This notebook is designed to be self-contained and reproducible, with all required tools organized in a `tools/` directory and file paths defined relative to the repository structure.

---

## Environment and Tools Setup

This pipeline relies on several external command-line tools. Ensure that the following are installed and available in your system `PATH`:

- **SRA Toolkit** (`prefetch`, `fasterq-dump`) – for downloading sequencing data from NCBI SRA  
- **pigz** – for parallel compression of FASTQ files  

---

## Tool Configuration

The notebook will attempt to automatically detect required tools in your environment. If they are not found, install them using the following resources:

- SRA Toolkit: official installation guide  
- pigz: install via your package manager (e.g., `sudo apt install pigz` on Linux or `brew install pigz` on macOS)

Alternatively, you may manually specify tool locations:

```{r}
SRA_TOOLKIT_PATH <- "path/to/sratoolkit"  # optional
PIGZ_PATH <- "path/to/pigz"              # optional
```

## Example Dataset

This workflow uses the public dataset GSE326480 as an example. Strandedness was not specified in metadata. Therefore, analysis assumes unstranded libraries. One control sample (SRR37862073) exhibited differences in sequencing strategy relative to the remaining samples (single-end/strand-specific vs paired-end). To ensure consistency across samples and minimize technical variability, this sample was excluded from downstream analyses. Users are advised to verify library type and strandedness for their own datasets prior to analysis.

## References
Kim, D., Paggi, J.M., Park, C. et al. Graph-based genome alignment and genotyping with HISAT2 and HISAT-genotype. Nat Biotechnol 37, 907–915 (2019). https://doi.org/10.1038/s41587-019-0201-4

Gabay M, Li Y, Felsher DW. MYC activation is a hallmark of cancer initiation and maintenance. Cold Spring Harb Perspect Med. 2014 Jun 2;4(6):a014241. doi: 10.1101/cshperspect.a014241. PMID: 24890832; PMCID: PMC4031954.

In [2]:
import os
import shutil

# Relative paths from the notebook
repo_root = os.path.abspath(os.path.join("..", ".."))  # two levels up
tools_bin = os.path.join(repo_root, "tools", "sratoolkit.3.2.1-ubuntu64", "bin")
pigz_bin = os.path.join(repo_root, "tools", "pigz-2.8")

# Append to PATH
os.environ["PATH"] += os.pathsep + tools_bin + os.pathsep + pigz_bin

# Detect tools
fasterq_path = shutil.which("fasterq-dump")
pigz_path = shutil.which("pigz")

# Raise clear error if tools are missing
if not fasterq_path:
    raise RuntimeError("fasterq-dump not found! Check that sratoolkit is in tools folder.")
if not pigz_path:
    raise RuntimeError("pigz not found! Check that pigz is in tools folder.")

print("Detected tools:")
print("fasterq-dump:", fasterq_path)
print("pigz:", pigz_path)

Detected tools:
fasterq-dump: /home/andreaciarletto/projects/genomics-toolkit/tools/sratoolkit.3.2.1-ubuntu64/bin/fasterq-dump
pigz: /home/andreaciarletto/projects/genomics-toolkit/tools/pigz-2.8/pigz


In [7]:
# Downloads Fastq files from SRA using SRR accession list
!scripts/download_fastq.sh


=== Processing SRR37862068 ===
2026-04-22T18:14:54 prefetch.3.2.1: 1) Resolving 'SRR37862068'...
2026-04-22T18:14:55 prefetch.3.2.1: Current preference is set to retrieve SRA Normalized Format files with full base quality scores
2026-04-22T18:14:55 prefetch.3.2.1: 1) Downloading 'SRR37862068'...
2026-04-22T18:14:55 prefetch.3.2.1:  SRA Normalized Format file is being retrieved
2026-04-22T18:14:55 prefetch.3.2.1:  Downloading via HTTPS...
2026-04-22T18:17:08 prefetch.3.2.1:  HTTPS download succeed
2026-04-22T18:17:14 prefetch.3.2.1:  'SRR37862068' is valid: 3141333726 bytes were streamed from 3141323443
2026-04-22T18:17:14 prefetch.3.2.1: 1) 'SRR37862068' was downloaded successfully
2026-04-22T18:17:14 prefetch.3.2.1: 1) Resolving 'SRR37862068's dependencies...
2026-04-22T18:17:14 prefetch.3.2.1: 'SRR37862068' has 0 unresolved dependencies
spots read      : 22,984,319
reads read      : 45,968,638
reads written   : 45,968,638
=== Processing SRR37862069 ===
2026-04-22T18:20:19 prefetch.3.

In [12]:
# Run FastQC to identify poor quality reads and adapter sequence content
!scripts/run_fastqc.sh

Running FastQC on raw_data/fastq/SRR37862068_1.fastq.gz ...
Started analysis of SRR37862068_1.fastq.gz
Approx 5% complete for SRR37862068_1.fastq.gz
Approx 10% complete for SRR37862068_1.fastq.gz
Approx 15% complete for SRR37862068_1.fastq.gz
Approx 20% complete for SRR37862068_1.fastq.gz
Approx 25% complete for SRR37862068_1.fastq.gz
Approx 30% complete for SRR37862068_1.fastq.gz
Approx 35% complete for SRR37862068_1.fastq.gz
Approx 40% complete for SRR37862068_1.fastq.gz
Approx 45% complete for SRR37862068_1.fastq.gz
Approx 50% complete for SRR37862068_1.fastq.gz
Approx 55% complete for SRR37862068_1.fastq.gz
Approx 60% complete for SRR37862068_1.fastq.gz
Approx 65% complete for SRR37862068_1.fastq.gz
Approx 70% complete for SRR37862068_1.fastq.gz
Approx 75% complete for SRR37862068_1.fastq.gz
Approx 80% complete for SRR37862068_1.fastq.gz
Approx 85% complete for SRR37862068_1.fastq.gz
Approx 90% complete for SRR37862068_1.fastq.gz
Approx 95% complete for SRR37862068_1.fastq.gz
Analy

In [8]:
# Run MultiQC to summarize FastQC reports
!scripts/run_multiqc.sh


Running MultiQC on qc/fastqc ...

/// ]8;id=654622;https://multiqc.info\MultiQC]8;;\ 🌏 v1.30

     version_check | MultiQC Version v1.34 now available!
       file_search | Search path: /home/andreaciarletto/projects/genomics-toolkit/examples/bulk_rnaseq/qc/fastqc
         searching | ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 22/22  stqc/SRR37862073_fastqc.html.html
            fastqc | Found 11 reports
     write_results | Data        : qc/multiqc/multiqc_data
     write_results | Report      : qc/multiqc/multiqc_report.html
           multiqc | MultiQC complete
=== MultiQC complete ===
Results in: qc/multiqc


### Quality Control Summary

The MultiQC report indicates that all samples have high-quality (Phred) scores above => 35. Adapter contamination was not detected. 

Less than 1% of reads made up of overrepresented sequences. 

In [ ]:
# Optional step: Run Cutadapt to trim adapters and low-quality bases only if FastQC indicates adapter contamination 
# or overrepresented sequences that could affect downstream analysis

!scripts/run_cutadapt.sh

In [ ]:
# Align paired-end RNA-seq reads with HISAT2 and output sorted BAM files
!scripts/run_hisat2.sh

HISAT2 index already exists. Skipping download.
/home/andreaciarletto/projects/genomics-toolkit/scripts/../examples/bulk_rnaseq/alignments/SRR37862068.sorted.bam already exists. Skipping alignment.
/home/andreaciarletto/projects/genomics-toolkit/scripts/../examples/bulk_rnaseq/alignments/SRR37862069.sorted.bam already exists. Skipping alignment.
/home/andreaciarletto/projects/genomics-toolkit/scripts/../examples/bulk_rnaseq/alignments/SRR37862070.sorted.bam already exists. Skipping alignment.
/home/andreaciarletto/projects/genomics-toolkit/scripts/../examples/bulk_rnaseq/alignments/SRR37862071.sorted.bam already exists. Skipping alignment.
/home/andreaciarletto/projects/genomics-toolkit/scripts/../examples/bulk_rnaseq/alignments/SRR37862072.sorted.bam already exists. Skipping alignment.
=== Alignment complete. BAM files in /home/andreaciarletto/projects/genomics-toolkit/scripts/../examples/bulk_rnaseq/alignments ===


In [13]:
# Download GTF annotation if needed and run featureCounts.sh
!scripts/run_featurecounts.sh

--2026-04-22 15:28:08--  ftp://ftp.ensembl.org/pub/release-109/gtf/homo_sapiens/Homo_sapiens.GRCh38.109.gtf.gz
           => ‘/home/andreaciarletto/projects/genomics-toolkit/examples/bulk_rnaseq/scripts/../examples/bulk_rnaseq/reference/Homo_sapiens.GRCh38.109.gtf.gz’
Resolving ftp.ensembl.org (ftp.ensembl.org)... 193.62.193.169
Connecting to ftp.ensembl.org (ftp.ensembl.org)|193.62.193.169|:21... connected.
Logging in as anonymous ... Logged in!
==> SYST ... done.    ==> PWD ... done.
==> TYPE I ... done.  ==> CWD (1) /pub/release-109/gtf/homo_sapiens ... done.
==> SIZE Homo_sapiens.GRCh38.109.gtf.gz ... 54258835
==> PASV ... done.    ==> RETR Homo_sapiens.GRCh38.109.gtf.gz ... done.
Length: 54258835 (52M) (unauthoritative)

Homo_sapiens.GRCh38 100%[===================>]  51.75M  14.9MB/s    in 3.6s    

2026-04-22 15:28:13 (14.2 MB/s) - ‘/home/andreaciarletto/projects/genomics-toolkit/examples/bulk_rnaseq/scripts/../examples/bulk_rnaseq/reference/Homo_sapiens.GRCh38.109.gtf.gz’ saved

In [9]:
# Get SRA metadata from Run Selector (https://www.ncbi.nlm.nih.gov/)
!Rscript scripts/get_metadata.R


Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union

Detected samples:
[1] "SRR37862068" "SRR37862069" "SRR37862070" "SRR37862071" "SRR37862072"

Final sample info:
                 sample condition     treatment batch    cell_line
SRR37862068 SRR37862068     acss2   ACSS2 shRNA  Rep3 MDA-MB-231BR
SRR37862069 SRR37862069     acss2   ACSS2 shRNA  Rep2 MDA-MB-231BR
SRR37862070 SRR37862070     acss2   ACSS2 shRNA  Rep1 MDA-MB-231BR
SRR37862071 SRR37862071   control Control shRNA  Rep3 MDA-MB-231BR
SRR37862072 SRR37862072   control Control shRNA  Rep2 MDA-MB-231BR


In [15]:
# Differential expression analysis using DESeq2
!Rscript scripts/run_deseq2.R



Loading required package: S4Vectors
Loading required package: stats4
Loading required package: BiocGenerics
Loading required package: generics

Attaching package: ‘generics’

The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union


Attaching package: ‘BiocGenerics’

The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs

The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min


Attaching package: ‘S4Vectors’

The following object is masked from ‘package:utils’:

    findMatches

The following object

# Results and Discussion

This RNA-seq analysis pipeline was tested using dataset GSE326480 (GSM9632583) submitted by Drexel University College of Medicine. 

In this study, MDA-MB-231BR breast cancer cells stably expressing either control shRNA (shSCR) or ACSS2-targeting shRNA (shACSS2), with three biological replicates per condition were analyzed. After alignment and count generation, a total of 16,875 protein-coding genes were detected with at least one mapped read, consistent with the previously reported value of ~17,721 genes.To improve statistical robustness, genes with low read support were filtered prior to analysis, retaining 14,762 genes with total counts ≥ 10 across all samples for downstream differential expression testing.

**Differential gene expression analysis reveals a moderate transcriptional response to ACSS2 knockdown**

Differential expression analysis using DESeq2 identified 231 significantly differentially expressed genes (adjusted p-value < 0.05), including 123 upregulated and 108 downregulated genes in ACSS2-depleted cells. This indicates that ACSS2 knockdown induces a moderate and specific transcriptional response rather than global transcriptome disruption.

**ACSS2 knockdown suppresses proliferative and metabolic gene programs**

Gene set enrichment analysis (FGSEA) using Hallmark gene sets revealed significant downregulation of multiple proliferation-associated pathways in shACSS2 cells.

The most strongly negatively enriched pathway was HALLMARK_MYC_TARGETS_V1 (NES = −1.65, padj = 0.0017), followed by HALLMARK_MYC_TARGETS_V2 (NES = −1.51, padj = 0.041). Additional negatively enriched pathways included E2F targets, DNA repair, oxidative phosphorylation, and fatty acid metabolism, although some of these did not reach statistical significance after multiple testing correction.

These results indicate suppression of MYC- and E2F-driven transcriptional programs, suggesting reduced proliferative and biosynthetic activity following ACSS2 depletion.

**ACSS2 knockdown activates hypoxia-associated and glycolytic signaling pathways**

In contrast, positively enriched pathways included HALLMARK_HYPOXIA (NES = 1.54, padj = 0.041), which was the most significantly upregulated gene set.

Additional positively enriched pathways included glycolysis, angiogenesis, and IL6/JAK/STAT3 signaling, although these did not reach statistical significance after correction.

Overall, these results suggest activation of a hypoxia-like transcriptional program, accompanied by a trend toward increased glycolytic metabolism and stress-response signaling.

**Figures supporting differential expression and pathway analysis**

Principal component analysis (PCA) of variance-stabilized counts demonstrated clear separation between shSCR and shACSS2 samples, indicating distinct global transcriptional profiles. Differential expression results were visualized using an MA plot, showing the distribution of gene expression changes across all detected genes. A volcano plot highlighted significantly upregulated and downregulated genes based on adjusted p-value and log2 fold-change thresholds. Finally, pathway-level changes identified by FGSEA are summarized using enrichment plots and ranked pathway barplots, highlighting coordinated transcriptional shifts following ACSS2 knockdown.

**Summary**
Pathway-level analysis revealed two opposing transcriptional trends following ACSS2 knockdown:

Downregulated pathways: MYC targets, E2F targets, DNA repair, oxidative phosphorylation, fatty acid metabolism
Upregulated pathways: hypoxia signaling, glycolysis, angiogenesis, inflammatory response pathways

Together, these results suggest coordinated remodeling of metabolic and proliferative gene programs in response to ACSS2 depletion.

# Discussion

This study shows that ACSS2 knockdown in MDA-MB-231BR metastatic breast cancer cells leads to a coordinated transcriptional reprogramming, with 231 genes significantly altered out of 14,762 expressed genes after filtering low-abundance transcripts. Pathway analysis revealed suppression of MYC- and E2F-driven proliferative programs, along with reduced oxidative phosphorylation and fatty acid metabolism, suggesting impaired biosynthetic and mitochondrial metabolic activity. In contrast, ACSS2 depletion activated hypoxia-associated gene expression, increased glycolytic signaling, and enriched inflammatory and stress-response pathways such as IL6/JAK/STAT3 and angiogenesis. Together, these results support a model in which ACSS2 helps sustain oxidative, lipid-associated metabolism and proliferative signaling in breast cancer cells, while its loss triggers a shift toward a hypoxia-like, glycolysis-dependent, and stress-adaptive state, highlighting ACSS2 as a key regulator of metabolic flexibility and tumor-associated transcriptional programs.

A limitation of the pathway-level ranking approach used in this pipeline (FGSEA on a ranked gene list) is that results depend on the chosen ranking metric, which in this case combines log2 fold change with statistical significance. This can bias enrichment toward genes that are both strongly and significantly differentially expressed, potentially amplifying certain pathway signals while underrepresenting biologically relevant but modest changes. In addition, gene set enrichment does not account for gene–gene interactions or directionality beyond aggregate scoring, and therefore should be interpreted as hypothesis-generating rather than definitive evidence of pathway activation or suppression.

